In [0]:
raw_df = spark.table("career_flow_engine.bronze.careerflow_raw")
display(raw_df.count())

In [0]:
desc_df = raw_df.select("job_id","raw_description","scraped_at","seniority_level")
display(desc_df.count())

In [0]:
desc_df = raw_df.select("job_id","raw_description")
duplicate_df = desc_df.groupBy("job_id", "raw_description").count().filter("count > 1")
display(duplicate_df.count())

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

desc_df = raw_df.select("job_id", "raw_description", "scraped_at")
window_spec = Window.partitionBy("job_id", "raw_description").orderBy(col("scraped_at").desc())
deduped_df = desc_df.withColumn("row_num", row_number().over(window_spec)).filter("row_num = 1").drop("row_num")
display(deduped_df.count())

In [0]:
from pyspark.sql.functions import col, countDistinct, trim

# Count distinct values in each grouping column
summary = desc_df.select(
    countDistinct('job_id').alias('distinct_job_id'),
    countDistinct('raw_description').alias('distinct_raw_description')
)
display(summary)

# Show sample values and null/whitespace counts
for col_name in ['job_id', 'raw_description']:
    print(f"\nColumn: {col_name}")
    desc_df.select(col_name).groupBy(col_name).count().orderBy('count', ascending=False).show(5)
    null_count = desc_df.filter(col(col_name).isNull()).count()
    print(f"Null count: {null_count}")
    whitespace_count = desc_df.filter(trim(col(col_name)) == '').count()
    print(f"Whitespace-only count: {whitespace_count}")